# Technical Assignment (NLP / IR / Gen AI / Semantic Search)
**Objective:**  The purpose of this assignment is to assess your skills in Natural Language
 Processing (NLP), Information Retrieval (IR), Generative AI, and Semantic Search. You will
 need to source an open dataset, perform an in-depth analysis, build a model, and derive
 actionable insights.

 **Assignment Description:** You are tasked with sourcing an open dataset related to textual data
 and using it to address a problem of your choice within the realm of NLP, IR, Gen AI, or
 Semantic Search. The focus can be on tasks such as text classification, summarization, topic
 modeling, question answering, media recommendations, or semantic search.

## Table of Contents
 1. [Dataset Sourcing](#dataset)
 2. [Problem Definition](#problem-definition)
 3. [Approach and Pipeline](#approach-and-pipeline)
 4. [Implementation](#implementation)
 5. [Evaluation](#evaluation)
 6. [Recommendations](#recommendations)

### Dataset Sourcing <a class="anchor" id="dataset"></a>

For this assignment I've selected the **MIcrosoft News Dataset (MIND)** to build a retrieval-augmented generation (RAG) system. MIND is a large-scale dataset of 160k English news articles and 15 million user behaviour logs that was curated specifically for recommendation tasks. Given the time limitation, I've opted for a smaller version of the MIND dataset with randomly sampled 50,000 users and their behaviour logs and corresponding news.

**Main reasons behind the choice** of the MIND dataset:
- The dataset's rich textual content including title, abstract, body, category and entities, makes it suitable for NLP-related tasks, e.g. semantic search, information retrieval and question answering.
- The dataset is not limited to one specific domain and covers a diversity of topics. In this case study, prompted by Dow Jones' focus on business news and financial information, I limited the scope to "news" and "finance" categories only.
- The dataset has user interaction data that allows to explore ways to evaluate retrieval quality.

### Problem Definition <a class="anchor" id="problem-definition"></a>

**Problem:** With the current volume of available content and news, finding and processing relevant information can be both extremely time-consuming and overwhelming. And even more so in case of time-sensitive material like news articles.

This is a significant problem, given that access to reliable and timely news and data is crucial for data-driven decision making.

**How a Q&A RAG could add to solution**: Having a tool that retrieves relevant articles given the user's question, summarizes them and outputs a document-based concise reply would save time and lower the cognitive overload.

### Approach and Pipeline <a class="anchor" id="approach-and-pipeline"></a>

**Approach:**
This project is a Retrieval-Augmented Generation (RAG) Q&A based on news articles. It offers a conversational reply with an answer to user's question and lists a selection of news articles that can be seen as recommendations: each has a url link and a short summary.

**Tools & frameworks:** For its development I've used the following models, tools and frameworks:

- **LangChain** as a framework to manage document preparation, embedding generation and vector store indexing, as well as retrieval and integrating LLMs for zero-shot classification, text generation and evaluation
- **Pinecone** as vector database used for storing and retrieval of documents (in this case news articles) and its Python **pinecone** library
- **Sentence Transformers** library to calculate dense vectors
- Hybrid search using **dense** and **sparse vectors** calculated with 
    - [finance-embeddings-investopedia](https://huggingface.co/FinLang/finance-embeddings-investopedia) for finance application trained by the FinLang team with the embedding size of 768 for embedding dense vectors
    - Sparse Lexical and Expansion model, or [SPLADE](https://github.com/naver/splade) for getting BERT-enriched sparse vectors
- **Llama 3.2 3B** model used locally via **Ollama** for Zero-shot classification and RAG Evaluation
- OpenAI's **GPT-4o mini** model used to generate RAG reply (a combination of structured and conversational response) based on user queries
- **pydantic** library to get structured outputs from the language models
- **wordninja** library to split concatenated words in subcategories labels

**Main Distinctive Features**:
- **Zero-shot classification with LLM** for metadata filtering at retrieval stage

    Zero-shot classification is carried out utilizing **Llama 3.2 3B** model via **Ollama** with structured output via Pydantic BaseModel. It is applied to infer from user queries key metadata like subcategories and named entities that can be used for metadata filtering when retrieving relevant documents from Pinecone index. It helps improve search relevance and reduces unnecessary processing. However, given that Pinecone (unlike, e.g. Elasticsearch) doesn't allow to use boosting or "should"-type filtering, in order to counteract potentially limiting effects of the filters, the filtering is done in three steps with gradually less strict filters to allow for broader spectrum of candidates.

- **Hybrid search** (dense and sparse embeddings) using **Pinecone** index

    The retrieval step is performed using hybrid vector search. The choice of **Pinecone** as vector database is due to both its low-latency retrieval and the option of using SPLADE for hybrid search. Combining dense vectors (in our case, **finance-embeddings-investopedia** with 768 dimensions) and **SPLADE** (Sparse Lexical and Expansion) vectors enhances RAG by leveraging the strengths of both semantic and lexical retrieval. Dense vectors capture deep contextual meanings, improving recall for queries with complex or implicit intent, while SPLADE expands sparse keyword representations, ensuring precise keyword matching.

    The choice of **finance-embeddings-investopedia** model was dictated by the chosen categories of the news articles: news and finance. This model was finetuned on Investopedia embedding dataset for tasks like clustering or semantic search in RAG.

- **Conversational and structured AI output**

    The final AI output generated with **OpenAI's GPT-4o mini model** combines a conversational user-friendly reply and a structured part with detailed information about each article used for the reply and thus recommended to the user (title, url, summary).

- **Evaluation chain to evaluate RAG's performance**
    
    The output of the RAG is evaluated with the use of **LangChain chain** with the use of **Llama 3.2 3B** model. The output  of the evaluation gives a verdict on the document relevancy, groundedness of the reply, overall score of the output and a brief reasoning.


**Pipeline**:
1. **Dataset EDA, Cleanup and Feature Engineering**:
    - Basic EDA and cleanup (nulls, duplicates)
    - Splitting subscategories tags into meaningful words using *wordninja* library
    - Processing json-like fields with named entities extracted from news titles and abstracts to get a list of labels with high level of confidence
    - Processing interaction data and merging it with news to prepare a dataframe for retrieval evaluation based on browsing history and generated user query
2. **Indexing**: News articles data is embedded into dense and sparse vectors, and sent to Pinecone along with metadata
3. **User query processing**: The zero-shot classifier extracts from user's query metadata that can be used to construct filters at retrieval stage
4. **Retrieval**: The system retrieves documents about the most relevant videos for the user's query using metadata filters (3 sets of filters of varying strictness), semantic search with dense vectors and lexical search with enriched sparse vectors
5. **Generation**: The system takes a user query and a list of retrieved documents, and outputs a conversational response and a curated list of the top 3-5 sources with their urls and short summary
6. **Evaluation**:
    - Retrieval Evaluation with Precision@K and Recall@K
        - Generating a potential user query for a given impression based on user's latest browsing history
        - Retrieving top 100 documents with this query
        - Calculating Precision@K and Recall@K for retrieved documents taking user's positive impressions as reference
    - RAG evaluation using LLM
    
        Another LLM chain takes the user query, retrieved documents and RAG reply as input and evaluates the document relevancy, groundedness of the reply and overall quality of the RAG output.

### Implementation <a class="anchor" id="implementation"></a>

In [11]:
#libraries
import os
import pandas as pd 
import numpy as np
import pinecone
import time
import torch
from sentence_transformers import SentenceTransformer
from pinecone_text.sparse import SpladeEncoder
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from typing import List
from pydantic import BaseModel as PydanticBaseModel
from pydantic import Field
from tenacity import(
    retry,
    stop_after_attempt,
    wait_random_exponential
)
import wordninja
import itertools
import json
from functools import reduce
import pickle
#options
device = 'cuda' if torch.cuda.is_available() else 'cpu'
pd.set_option('chained_assignment',None)

#### Load and preprocess the data

##### News

In [2]:
#load the news data
news_df = pd.read_csv('data/train/news.tsv', sep='\t', header=None, names=['id', 'category', 'subcategory', 'news_title', 'news_abstract', 'url', 'entities_title', 'entities_abstract'])

In [3]:
news_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51282 entries, 0 to 51281
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 51282 non-null  object
 1   category           51282 non-null  object
 2   subcategory        51282 non-null  object
 3   news_title         51282 non-null  object
 4   news_abstract      48616 non-null  object
 5   url                51282 non-null  object
 6   entities_title     51279 non-null  object
 7   entities_abstract  51278 non-null  object
dtypes: object(8)
memory usage: 3.1+ MB


In [5]:
#check available categories
news_df.category.value_counts()

category
news             15774
sports           14510
finance           3107
foodanddrink      2551
lifestyle         2479
travel            2350
video             2068
weather           2048
health            1885
autos             1639
tv                 889
music              769
movies             606
entertainment      587
kids                17
middleeast           2
northamerica         1
Name: count, dtype: int64

In [6]:
#leave only news and finance
news_specialized_df = news_df[news_df['category'].isin(['news', 'finance'])]

In [7]:
news_specialized_df.shape

(18881, 8)

In [9]:
#check for nulls
news_specialized_df.isna().sum()

id                     0
category               0
subcategory            0
news_title             0
news_abstract        630
url                    0
entities_title         0
entities_abstract      1
dtype: int64

In [12]:
#given that we'll be using news abstract as document body and the percentage of nulls is low, let's drop all rows with null values
news_specialized_df.dropna(inplace=True)

In [ ]:
#check for duplicates
news_specialized_df.duplicated().any()

False

In [23]:
news_specialized_df.head(2)

,id,category,subcategory,news_title,news_abstract,url,entities_title,entities_abstract
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
7,N59295,news,newsworld,Chile: Three die in supermarket fire amid prot...,Three people have died in a supermarket fire a...,https://assets.msn.com/labs/mind/AAJ43pw.html,"[{""Label"": ""Chile"", ""Type"": ""G"", ""WikidataId"":...","[{""Label"": ""Santiago"", ""Type"": ""G"", ""WikidataI..."


The subcategories appear as tags or concatenated words. In order to use them later for filtering, we need to split them into real words.

One way to do it is to use a lightweight probabilistic segmentation with wordninja library. It seems to be the best solution here given that the subcategories use rather common and potentially easily segmentable words.

In [25]:
#using lightweight probabilistic segmentation with wordninja
news_specialized_df.loc[:, "subcategory"] = news_specialized_df.loc[:,"subcategory"].apply(lambda x: ' '.join(wordninja.split(x)))

The named entities already extracted by the authors of the dataset also need transforming in order to be used. Let's combine labels from both titles and abstacts that have a confidence score greater than 95%.

In [26]:
def parse_and_filter_entities(entities_string):
    try:
        entities = json.loads(entities_string)
        labels = [item['Label'] for item in filter(lambda a: a['Confidence'] > 0.95, entities)]
    except:
        labels = []
    return labels

#parse entities fields as json and combine those for titles and abstracts in one field
news_specialized_df["entities"] = news_specialized_df[['entities_title', 'entities_abstract']]\
                                .apply(lambda x: list(set(reduce(lambda a, b: a+b, list(map(parse_and_filter_entities, x))))), axis=1)

##### Interaction data

In [16]:
#load users behaviour data
interactions_df = pd.read_csv('data/train/behaviors.tsv', sep='\t', header=None, names=['impression_id', 'user_id', 'time', 'history', 'impressions'])

In [17]:
interactions_df.shape

(156965, 5)

In [18]:
interactions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156965 entries, 0 to 156964
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   impression_id  156965 non-null  int64 
 1   user_id        156965 non-null  object
 2   time           156965 non-null  object
 3   history        153727 non-null  object
 4   impressions    156965 non-null  object
dtypes: int64(1), object(4)
memory usage: 6.0+ MB


In [19]:
#check nulls
interactions_df.isna().sum()

impression_id       0
user_id             0
time                0
history          3238
impressions         0
dtype: int64

In [20]:
#given that we'll need history to create potential user queries, and the percentage of nulls is low, let's drop all rows with null values
interactions_df.dropna(inplace=True)

In [21]:
#check duplicates
interactions_df.duplicated().any()

False

One of potential ways of evaluating the retrieval quality with this dataset is generating potential user queries based on user latest browsing history (assuming that they would continue navigating the same topic), retrieving documents based on these queries and calculating precision@K and recall@K taking positive impressions as reference.
In order to prepare the data for potential user query generation and metrics calculations we need to carry out a series of transformations on the interactions dataframe.

In [24]:
interactions_df.head(2)

,impression_id,user_id,time,history,impressions
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...


In [27]:
#transform a string with impressions into a list
interactions_df['impressions_list'] = interactions_df.impressions.str.split(' ')

In [28]:
#extract news_id and click / non click as separate fields
interactions_df['impression_news_id'] = interactions_df.impressions_list.apply(lambda x: [item[:-2] for item in x])
interactions_df['click'] = interactions_df.impressions_list.apply(lambda x: [item[-1] for item in x])

In [ ]:
#get a new dataset with click / non click as level of aggregation
for_testing = interactions_df.explode(['impression_news_id', 'click'])

In [30]:
#leave only impressions with positive clicks from the mind small database with business and news articles only
for_testing = for_testing.merge(news_specialized_df, how='inner', left_on='impression_news_id', right_on='id').query('click=="1"')

In [31]:
#get top 100 impressions sorted by time and number of clicked news
top_impressions = for_testing.groupby('impression_id', as_index=False)\
                    .agg({'impression_id': 'max', 'click': 'size', 'time': 'max', 'impression_news_id': lambda x: list(x), 'category': lambda x: set(x), 'subcategory': lambda x: set(x)}).sort_values(['click', 'time'], ascending=False).head(100)

In [32]:
#merge with interactions to get browsing history for top 100 impression
for_user_queries = top_impressions.merge(interactions_df, how='inner', on='impression_id')[['impression_id', 'history']]

In [33]:
#get articles' details for the browsing history
for_user_queries['browsing'] = for_user_queries['history'].str.split(' ')
for_user_queries = for_user_queries.explode('browsing').merge(news_df, how='inner', left_on='browsing', right_on='id')

In [ ]:
#get a string of news titles from the last three (maximum) consulted articles before the query that will be used later for query generation
user_queries = for_user_queries.groupby('impression_id', as_index=False).agg({'impression_id': 'max', 'news_title': lambda x: '; '.join(x[-min(3, len(x)):])})[['impression_id', 'news_title']]

#### Build the index

In [105]:
#instantiate Pinecone client
pc = pinecone.Pinecone(api_key=os.getenv('PINECONE_API'), pool_threads=6)

In [107]:
#create index if it doesn't exist
index_name = 'mind-small'
if index_name not in pc.list_indexes().names():
        print(f"Creating {index_name} index.")
        pc.create_index(
          name=index_name,
          metric="dotproduct",
          dimension=768,
          spec=pinecone.ServerlessSpec(
                                    cloud="aws",
                                    region="us-east-1"
                                  )
        )
        while not pc.describe_index(index_name).status['ready']:
            time.sleep(1)

In [108]:
#connect to index
index = pc.Index(index_name)
print(f"Connected to {index_name}.")

Connected to mind-small.


In [109]:
dense_vector_model = SentenceTransformer('FinLang/finance-embeddings-investopedia', device=device)
sparse_vector_model = SpladeEncoder(max_seq_length=512, device=device)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\eremi\miniconda3\envs\rag_langchain_ollama\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\eremi\.cache\huggingface\hub\models--FinLang--finance-embeddings-investopedia. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.20k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/796 [00:00<?, ?B/s]

... (10 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


In [16]:
data_records = news_specialized_df.to_dict('records')
data = []
for record in data_records:
    data.append({
            "id": record["id"],
            "context": f""""{record["news_title"]}\n{record["news_abstract"]}""",
            "title": record["news_title"],
            "category": record["category"],
            "subcategory": record["subcategory"],
            "entities": record["entities"],
            "sources": record["url"],
        })

In [ ]:
#calculate dense and sparse embeddings and load them asyncroniously to the vector database together with associated metadata
# it = iter(data)
it = pickle.load(open('iter_pickle.pkl','br'))

In [10]:
chunk = tuple(itertools.islice(it, 128))
while chunk:
    ids = [str(x["id"]) for x in chunk]
    contexts = [x["context"] for x in chunk]
    sources = [x['sources'] for x in chunk]
    titles = [x["title"] for x in chunk]
    categories = [x["category"] for x in chunk]
    subcategories = [x["subcategory"] for x in chunk]
    entities = [x["entities"] for x in chunk]
    
    dense_vectors = dense_vector_model.encode(contexts).tolist()
    sparse_vectors = sparse_vector_model.encode_documents(contexts)
    upsert_chunk = []

    for _id, dense_vector, sparse_vector, context, source, title, category, subcategory, entity in zip(ids,
                                                            dense_vectors,
                                                            sparse_vectors,
                                                            contexts,
                                                            sources,
                                                            titles,
                                                            categories,
                                                            subcategories,
                                                            entities
                                                            ):
        metadata = {
            "category": category,
            "subcategory": subcategory,
            "title": title,
            "context": context,
            "entities": entity,
            "source": source 
        }
        upsert_chunk.append(
            {
                "id": _id,
                "values": dense_vector,
                "sparse_values": sparse_vector,
                "metadata": metadata
            }
    )
    async_result = index.upsert(vectors=upsert_chunk, async_req=True)
    chunk = tuple(itertools.islice(it, 128))
    pickle.dump(it, open('iter_pickle.pkl', 'bw'))

#### Candidates Retrieval and Generating Final Recommendations <a class="anchor" id="rag"></a>

In [ ]:
def encode_scale_vectors(query, alpha):
        """Encode sparse and dense vectors, and apply hybrid vector scaling using a convex combination alpha * dense + (1 - alpha) * sparse
        Args:
        alpha (float): 0 == sparse only, 1 == dense only
        """
        #create sparse query vector
        sparse = sparse_vector_model.encode_queries(query)
        #create dense query vector
        dense = dense_vector_model.encode(query).tolist()
        #scale sparse and dense vectors
        hyb_sparse = {
            'indices': sparse['indices'],
            'values':  [v * (1 - alpha) for v in sparse['values']]
        }
        hyb_dense = [v * alpha for v in dense]
        return hyb_dense, hyb_sparse

In [112]:
def retrieve_from_pinecone(index, query, metadata_filter, top_k, alpha=0.5):
    hyb_dense, hyb_sparse = encode_scale_vectors(query=query, alpha=alpha)
    retrieved_docs = index.query(
        top_k=top_k,
        vector=hyb_dense,
        sparse_vector=hyb_sparse,
        include_metadata=True,
        include_values=False,
        filter=metadata_filter
    )
    documents = [Document(page_content=f"""News article: {item['metadata']['context']}\nTopic: {item['metadata']['subcategory']}\nKeywords{item['metadata']['entities']}""",
        metadata={"title": item['metadata']['title'],
        'source': item['metadata']['source']}) for item in retrieved_docs["matches"]]
    _ids = [item['id'] for item in retrieved_docs["matches"]]
    return documents, _ids

In [89]:
#analyse user query and generate retriever filter based on it

In [117]:
subcategories = news_specialized_df.subcategory.unique().tolist()
class BaseModel(PydanticBaseModel):
    class Config:
        arbitrary_types_allowed = True

class QueryDetails(BaseModel):
    category: list = Field(description="List of categories of news article the user is looking for",
                          enum=subcategories)
    named_entities: list = Field(description="List of named entitites.")

In [151]:
class QARetriever():
    def __init__(self):
        self.llm = ChatOllama(model= 'llama3.2', temperature=0.0)
        self.query_structured_llm = self.llm.with_structured_output(QueryDetails)
        self.query_classification_prompt = ChatPromptTemplate.from_template(
        """
        Analyse user's query and extract th following properties:
        1. List of categories of the news the user is interested in. Choose all the best matches from this list: {subcategories}.
        2. Main named entities and keyword phrases from the user's input.
        Use only data mentioned directly in the input. Output only those properties that you are absolutely sure of based on the user's input.
        Input: {input}
        """
        )

        self.query_classification_chain = self.query_classification_prompt | self.query_structured_llm

    @retry(wait=wait_random_exponential(min=1, max=30), stop=stop_after_attempt(3))

    def infer_query_metadata(self, query):
        return self.query_classification_chain.invoke({'subcategories': subcategories, 'input': query})

    def retrieve_docs(self, user_query, top_k=20):
        user_metadata = self.infer_query_metadata(query=user_query)
        user_category = user_metadata.category
        user_entities = user_metadata.named_entities

        metadata_filter = {
            "$and":
            
                    [
                        {"subcategory": {"$in": user_category}},
                        {"entitites": {"$in": user_entities}}]
        }

        metadata_light_filter = {
            "$or":
                    [
                        {"subcategory": {"$in": user_category}},
                        {"entitites": {"$in": user_entities}}]
        }

        #retrieve 3 batches of candidates with different filters
        retrieved_documents = []

        #with full filters applied
        retrieved_filtered_documents, _ids = retrieve_from_pinecone(index=index, query=user_query, metadata_filter = metadata_filter, top_k=top_k, alpha=0.5)
        retrieved_documents.extend(retrieved_filtered_documents)

        #with or-filters applied, and don't repeat already retrieved matches
        metadata_light_filter["id"] = {"$nin": _ids}
        top_k = max(3, top_k - len(_ids))
        retrieved_or_filter_documents, add_ids = retrieve_from_pinecone(index=index, query=user_query, metadata_filter = metadata_light_filter, top_k=top_k, alpha=0.5)
        _ids.extend(add_ids)
        retrieved_documents.extend(retrieved_or_filter_documents)

        #without filter and don't repeat already retrieved matches
        top_k = max(2, top_k - len(add_ids))
        metadata_minimal_filter = {"id": {"$nin": _ids}}
        retrieved_minimal_filter_documents, last_ids = retrieve_from_pinecone(index=index, query=user_query, metadata_filter = metadata_minimal_filter, top_k=top_k, alpha=0.5)
        retrieved_documents.extend(retrieved_minimal_filter_documents)
        _ids.extend(last_ids)

        return retrieved_documents, _ids


Pinecone doesn´t allow should or boost

In [119]:
class Articles(BaseModel):
    title: str = Field(description="Title of this particular relevant article")
    source: str = Field(description="Source url of this particular relevant article")
    summary: str = Field(description="Short summary of this particular relevant article")


class RecOutput(BaseModel):
    articles: List[Articles] = Field(description="A list of recommended articles used for the answer.")
    reply: str = Field(description="A conversational output with an answer to the user's question.")

In [120]:
class QAGeneration():
    def __init__(self):
        self.rag_llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.2, frequency_penalty = 0.1, api_key=os.getenv('OPENAI_API_KEY'))

        self.system_prompt = (
        """You are an assistant for question-answering tasks and recommending news articles. Use the following pieces of retrieved context to answer the question. {context}. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
        Return two things:
        1. A conversational output where you give an anwer to the user's question in one short paragraph.
        2. A list of documents, that you used to answer the question. Each document should be represnted as a json with title, source and summary. Make sure that all elements in one json come from the same document.
        """
        )

        self.qa_prompt = ChatPromptTemplate.from_messages(
            [
                ("system", self.system_prompt),
                ("human", "{input}"),
            ]
        )

        self.rag_chain_from_docs = (
                    {
                        "input": lambda x: x["input"],
                        "context": lambda x: x["context"]
                    }
                    | self.qa_prompt
                    | self.rag_llm.with_structured_output(RecOutput)
                )
        self.chain = RunnablePassthrough.assign(answer=self.rag_chain_from_docs)
        
    def get_answer(self, user_query, retrieved_documents):
        rag_output = self.chain.invoke({"input": user_query, "context": retrieved_documents})
        conv_reply = rag_output['answer'].reply
        articles = rag_output['answer'].articles
        return conv_reply, articles

In [152]:
qa_retriever = QARetriever()
qa_generator = QAGeneration()

In [ ]:
user_query = input('Ask your question: ')

In [ ]:
retrieved_docs, _ids = qa_retriever.retrieve_docs(user_query=user_query)

In [62]:
conv_reply, articles = qa_generator.get_answer(user_query=user_query, retrieved_documents=retrieved_docs)

In [63]:
print(conv_reply)

Here are some articles about Goldman Sachs: one discusses their recent insider trading charges, another covers disappointing banking revenue and writedowns, and the last one details their retreat from Ken Fisher's firm after controversial remarks.


In [64]:
for article in articles:
    print(article.title, '\n', article.summary, '\n', article.source, '\n\n')

Goldman's Unwelcome Streak: A String of Insider Trading Charges 
 Goldman Sachs Group Inc. has built itself into a global dealmaking force whose bankers are often stitching together the biggest and most sensitive corporate transactions. 
 https://assets.msn.com/labs/mind/AAJ2bsv.html 


Goldman Sachs' quarter soured by disappointing banking revenue, writedowns for Uber and other investments 
 Goldman took a $267 million markdown on public investments in companies like Uber and Tradeweb Markets. 
 https://assets.msn.com/labs/mind/AAIOfrq.html 


Goldman Sachs Joins Retreat From Fisher's Firm After His Remarks 
 Goldman Sachs Group Inc. is joining the exodus from Ken Fisher's firm. 
 https://assets.msn.com/labs/mind/AAJjDGx.html 




### Evaluation <a class="anchor" id="evaluation"></a>

In [178]:
llm = ChatOllama(model='llama3.2', temperature=0.0)
query_prompt = ChatPromptTemplate.from_template(
"""
You have been given titles of articles from the user's browsing history - {input}.
Generate a search query this user is likely to use next. Output only a concise text of the query.
"""
)

query_chain = query_prompt | llm

def generate_user_query(browsing):
    llm_output = query_chain.invoke(input=browsing)
    return llm_output.content

In [142]:
user_queries['query'] = user_queries['news_title'].apply(generate_user_query)

In [143]:
user_queries.head(5)

,impression_id,news_title,query
0,4765,"Live score, updates: Colorado State basketball...","""Colorado State vs Duke basketball live score"""
1,6718,A couple was found buried on a Texas beach. Po...,"""Acetaminophen overdose symptoms"""
2,7519,"State Trooper Stops Banana Car Driver, Gives H...","""Good deeds by law enforcement officers"""
3,10071,'Sons Of Anarchy' Star Ron Perlman Files for D...,"""Ron Perlman news divorce"""
4,11643,Wrestling referee warned Rep. Jim Jordan about...,"""Jim Jordan Ohio State scandal"""


In [153]:
user_queries['retrieved_ids'] = user_queries['query'].head(10).apply(lambda x: qa_retriever.retrieve_docs(user_query=x, top_k=100)[1])

In [145]:
eval_df = user_queries.query('retrieved_ids.notna()').merge(top_impressions, how='inner', on='impression_id')

In [ ]:
#calculate Precision@K and Recall@K
def precision_recall(retrieved_docs, clicked_docs, k):
    if k > len(retrieved_docs):
        k = len(retrieved_docs)
    top_k_docs = retrieved_docs[:k]
    #how many of the top K documents are relevant
    clicked_in_top_k = sum([1 for doc in top_k_docs if doc in clicked_docs])
    #precision
    precision = clicked_in_top_k / k
    #recall
    recall = clicked_in_top_k / len(clicked_docs)
    return precision, recall

In [222]:
eval_df['metrics'] = eval_df[['retrieved_ids', 'impression_news_id']].apply(lambda x: precision_recall(retrieved_docs=x['retrieved_ids'], clicked_docs=x['impression_news_id'], k=100), axis=1)

In [227]:
eval_df['precision'] = eval_df['metrics'].apply(lambda x: x[0])
eval_df['recall'] = eval_df['metrics'].apply(lambda x: x[1])

In [213]:
class Evaluator(BaseModel):
    relevance: str = Field(description="Are the retrieved documents relevant to the question?", enum=["Yes", "No"])
    groundedness: str = Field(description="Is the answer grounded in the documents?", enum=["Yes", "No"])
    score: int = Field(description="a 1-5 score for the overall quality of the RAG output")
    reasoning: str = Field(description="Brief justification of the score")

In [216]:
eval_prompt = ChatPromptTemplate.from_template(
            """You are evaluating a response from a retrieval-augmented generation (RAG) system.
                Question: {query}
                Retrieved Documents:
                {contexts}
                RAG output:
                {prediction}

                Please answer the following:
                1. Are the retrieved documents relevant to the question? Output "Yes" or "No" only.
                2. Is the answer grounded in the documents? Output "Yes" or "No" only.
                3. Give a 1-5 score for the overall quality of the RAG output (1=Poor, 5=Excellent). Output only the score digit.
                4. Provide a brief justification of the score. Output one sentence maximum.

                """
        )

In [217]:
eval_chain = eval_prompt | llm.with_structured_output(Evaluator)

In [ ]:
def evaluate_strings(input, retrieved_docs, output):
        context = "\n\n".join(retrieved_docs)
        response = eval_chain.invoke({'query': input, 'contexts': context, 'prediction': output})
        return response

In [219]:
evaluate_strings(input='What is the mooon?', retrieved_docs="Moon is cheese", output="Moon is a planet")

Evaluator(relevance='Yes', groundedness='No', score=3, reasoning='The retrieved documents do not contain any information about the moon, but the RAG output is still relevant and accurate.')

### Recommendations <a class="anchor" id="recommendations"></a>
 ○ Offer recommendations for potential improvements or next steps.